# iNaturalist *Acacia* observations — Coimbra pilot

**Purpose.** Explore the citizen-science dataset on its own: how many
observations, over what time period, where, and with what quality/bias
indicators. Split out of the original
`01_problem_exploration/available_data_exploration.ipynb` notebook so
dataset-specific exploration lives in its own folder, per
`notebooks/README.md`.

**Scope.** Exploratory only — no modelling, no filtering decisions. This is
a *pilot download* for access validation (see `data/README.md`): genus
*Acacia*, iNaturalist place "Coimbra, PT" (resolves to the Coimbra
**district**, not just the city) — not a species/region decision.

See `../01_problem_exploration/available_data_exploration.ipynb` for the
file inventory across all datasets; see `../03_satellite/` for the TESSERA embeddings explored on
their own, and `../04_data_integration/` for the first feasibility check
linking the two.

Questions answered below:

1. How many observations are there?
2. What dates do they have / what is the temporal period?
3. Where are they located?
4. Are there several observations at the same place?
5. Are there areas with many observations and others with almost none?
6. What information comes with each observation?
7. Is there any indication of quality/confidence?

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.geometry import box

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"
INAT_CSV = RAW / "citizen_science" / "inaturalist_acacia_coimbra_pilot.csv"
TESSERA_DIR = RAW / "satellite" / "tessera_coimbra_pilot"
TESSERA_YEAR = 2024

# CRS used for metric computations (distances, grid cells). It is the CRS the
# TESSERA tiles are delivered in (see 03_satellite/), so observations and
# tile boundaries line up even though no embedding data is loaded here.
UTM = "EPSG:32629"

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 110

## 1. What information comes with each observation? (Q1, Q6)

In [ ]:
inat = pd.read_csv(INAT_CSV)
inat["observed_on"] = pd.to_datetime(inat["observed_on"], errors="coerce")
print(f"{len(inat):,} observations, {inat['id'].nunique():,} unique ids, {inat.shape[1]} columns")
inat.head()

In [ ]:
pd.DataFrame({
    "dtype": inat.dtypes.astype(str),
    "n_missing": inat.isna().sum(),
    "pct_missing": (inat.isna().mean() * 100).round(1),
    "n_unique": inat.nunique(),
    "example": inat.iloc[0],
})

Notes on the columns (from the iNaturalist API, as documented in `data/README.md`):

- `quality_grade` — iNaturalist's own data-quality label (`research`, `needs_id`, `casual`).
- `positional_accuracy` — observer-reported coordinate uncertainty, in metres. Missing for most records — check below.
- `coordinates_obscured` — true when the location was deliberately randomised (privacy or threatened taxon).
- `time_observed_at` — time of day; often missing, and not needed at TESSERA's annual resolution.
- There is **no abundance, cover, or absence information**: each row is a presence-only record.

## 2. Temporal period and date distribution (Q2)

In [ ]:
dates = inat["observed_on"]
print("missing dates:", dates.isna().sum())
print("earliest:", dates.min().date(), "| latest:", dates.max().date())

year_counts = dates.dt.year.value_counts().sort_index()
year_share = (year_counts / year_counts.sum() * 100).round(1)
pd.DataFrame({"n": year_counts, "pct": year_share}).T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
yc = year_counts[year_counts.index >= 2010]
axes[0].bar(yc.index.astype(int), yc.values, color="tab:green")
axes[0].set_title("Observations per year (>= 2010)")

mc = dates.dt.month.value_counts().sort_index()
axes[1].bar(mc.index.astype(int), mc.values, color="tab:green")
axes[1].set_xticks(range(1, 13))
axes[1].set_title("Observations per month (all years)")
plt.tight_layout()

### 3. Where are the observations? (Q3)

The iNaturalist query used the Coimbra **district** place; the TESSERA
pilot (see `../03_satellite/`) covers 12 tiles of 0.1° around the Coimbra
**municipality** — a different, smaller boundary (see `data/README.md`).
The tile boundaries are plotted below for context only, from the TESSERA
tile folder names — no embedding data is loaded in this notebook.

In [ ]:
inat_gdf = gpd.GeoDataFrame(
    inat, geometry=gpd.points_from_xy(inat["longitude"], inat["latitude"]), crs="EPSG:4326"
).to_crs(UTM)

tile_dirs = sorted((TESSERA_DIR / "global_0.1_degree_representation" / str(TESSERA_YEAR)).iterdir())
tile_centres = [tuple(float(v) for v in d.name.removeprefix("grid_").split("_")) for d in tile_dirs]
tiles = gpd.GeoDataFrame(
    {"tile": [d.name for d in tile_dirs]},
    geometry=[box(lon - 0.05, lat - 0.05, lon + 0.05, lat + 0.05) for lon, lat in tile_centres],
    crs="EPSG:4326",
).to_crs(UTM)

tessera_area = tiles.union_all()
inat_gdf["in_tessera"] = inat_gdf.within(tessera_area)
print(f"Observations inside downloaded TESSERA tiles: {inat_gdf['in_tessera'].sum():,} "
      f"of {len(inat_gdf):,} ({inat_gdf['in_tessera'].mean():.1%})")
inat[["latitude", "longitude"]].describe().loc[["min", "max", "mean"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
tiles.boundary.plot(ax=ax, color="black", lw=0.6)
for q, colour in [("research", "tab:green"), ("needs_id", "tab:orange"), ("casual", "tab:grey")]:
    sub = inat_gdf[inat_gdf["quality_grade"] == q]
    sub.plot(ax=ax, markersize=2, color=colour, alpha=0.5, label=f"{q} (n={len(sub):,})")
ax.set_title("iNaturalist Acacia observations; black grid = downloaded TESSERA tiles")
ax.legend(markerscale=5, fontsize=8)
ax.set_xlabel("Easting (m, UTM 29N)"); ax.set_ylabel("Northing (m)")
plt.tight_layout()

### 2.4 Several observations at the same place? (Q4)

Checked at increasing tolerance: identical coordinates, then the same 10 m
cell (= one TESSERA / Sentinel-2 pixel), 100 m and 1 km cells. Many records
sharing a pixel means they will collapse to the same feature vector once
satellite data are attached — relevant for dataset construction and for
spatial leakage between train and test.

In [ ]:
def grid_cell(gdf, size_m):
    return (gdf.geometry.x // size_m).astype(int).astype(str) + "_" + (gdf.geometry.y // size_m).astype(int).astype(str)

rows = [{
    "unit": "identical coordinates",
    "n_locations": inat.groupby(["latitude", "longitude"]).ngroups,
    "max_obs_per_location": inat.groupby(["latitude", "longitude"]).size().max(),
}]
for size in [10, 100, 1000]:
    counts = grid_cell(inat_gdf, size).value_counts()
    rows.append({"unit": f"{size} m cell", "n_locations": counts.size, "max_obs_per_location": counts.max()})
dup = pd.DataFrame(rows)
dup["obs_per_location_mean"] = (len(inat) / dup["n_locations"]).round(2)
dup["pct_obs_sharing_location"] = [
    round(inat.duplicated(["latitude", "longitude"], keep=False).mean() * 100, 1),
    *[round(grid_cell(inat_gdf, s).duplicated(keep=False).mean() * 100, 1) for s in [10, 100, 1000]],
]
dup

In [ ]:
same_place_same_day = inat.groupby(["latitude", "longitude", "observed_on"]).size()
print("Locations with >1 record on the same day (exact coords):", (same_place_same_day > 1).sum())
print("Records involved:", same_place_same_day[same_place_same_day > 1].sum())

### 2.5 Uneven sampling: dense areas vs. empty areas (Q5)

Counts per 1 km cell over the bounding box of the observations. The Lorenz
curve shows how concentrated observations are among the *occupied* cells; the
share of empty cells shows how much of the area has no record at all
(which must **not** be read as absence of *Acacia*).

In [ ]:
cell = 1000
xmin, ymin, xmax, ymax = inat_gdf.total_bounds
nx, ny = int(np.ceil((xmax - xmin) / cell)), int(np.ceil((ymax - ymin) / cell))
H, xedges, yedges = np.histogram2d(inat_gdf.geometry.x, inat_gdf.geometry.y, bins=[nx, ny],
                                   range=[[xmin, xmin + nx * cell], [ymin, ymin + ny * cell]])
occupied = np.sort(H[H > 0])[::-1]
print(f"1 km cells in bbox: {H.size:,} | with >=1 observation: {occupied.size:,} ({occupied.size / H.size:.1%})")
for k in [1, 10, 50]:
    print(f"Top {k:>2} cells hold {occupied[:k].sum() / occupied.sum():.1%} of observations")

fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1.6, 1]})
im = axes[0].imshow(np.ma.masked_equal(H.T, 0), origin="lower", cmap="viridis",
                    norm=plt.matplotlib.colors.LogNorm(),
                    extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]])
tiles.boundary.plot(ax=axes[0], color="red", lw=0.5)
fig.colorbar(im, ax=axes[0], label="observations per km² (log)")
axes[0].set_title("Observation density, 1 km cells (empty cells blank)")

asc = np.sort(occupied)
axes[1].plot(np.linspace(0, 1, asc.size + 1), np.r_[0, asc.cumsum() / asc.sum()], color="tab:green")
axes[1].plot([0, 1], [0, 1], ls="--", color="grey", lw=1)
axes[1].set_xlabel("cumulative share of occupied cells"); axes[1].set_ylabel("cumulative share of observations")
axes[1].set_title("Lorenz curve (occupied 1 km cells)")
plt.tight_layout()

### 2.6 Quality / confidence indicators (Q7)

In [ ]:
summary = pd.concat({
    "quality_grade": inat["quality_grade"].value_counts(),
    "taxon_rank": inat["taxon_rank"].value_counts(),
    "coordinates_obscured": inat["coordinates_obscured"].value_counts(),
}).to_frame("n")
summary["pct"] = (summary["n"] / len(inat) * 100).round(1)
summary

In [ ]:
acc = inat["positional_accuracy"]
print(f"positional_accuracy missing: {acc.isna().mean():.1%}")
bins = [0, 10, 30, 100, 1000, np.inf]
labels = ["<=10 m (within a pixel)", "10–30 m", "30–100 m", "100 m–1 km", ">1 km"]
pd.cut(acc.dropna(), bins=bins, labels=labels, include_lowest=True).value_counts().sort_index().to_frame("n")

Taxonomy: the genus query returns several species with different ecology
(and probably different spectral/phenological signatures). Records at genus
or subgenus level cannot be assigned to a species.

In [ ]:
species = inat.groupby("taxon_name").agg(
    n=("id", "size"),
    rank=("taxon_rank", "first"),
    pct_research=("quality_grade", lambda q: round((q == "research").mean() * 100, 1)),
).sort_values("n", ascending=False)
species["pct_of_total"] = (species["n"] / len(inat) * 100).round(1)
species

### 2.7 Observer concentration

Who produced the data matters for bias (RQ3): a single very active observer
imprints their own routes and habits on the spatial and temporal distribution.

In [ ]:
obs_per_user = inat["user_login"].value_counts()
print(f"{obs_per_user.size} distinct observers")
print(f"Top observer: {obs_per_user.iloc[0]:,} records ({obs_per_user.iloc[0] / len(inat):.1%})")
print(f"Top 10 observers: {obs_per_user.iloc[:10].sum() / len(inat):.1%}")
print(f"Observers with a single record: {(obs_per_user == 1).sum()}")

top_user = obs_per_user.index[0]
inat["observer_group"] = np.where(inat["user_login"] == top_user, "top observer", "all others")
inat_gdf["observer_group"] = inat["observer_group"].values
pd.crosstab(inat["observer_group"], inat["quality_grade"], margins=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
pd.crosstab(dates.dt.year, inat["observer_group"]).loc[2015:].plot.bar(stacked=True, ax=axes[0], width=0.8)
axes[0].set_title("Observations per year by observer group")
tiles.boundary.plot(ax=axes[1], color="black", lw=0.5)
for grp, colour in [("all others", "tab:blue"), ("top observer", "tab:orange")]:
    inat_gdf[inat_gdf["observer_group"] == grp].plot(ax=axes[1], markersize=2, alpha=0.5, color=colour, label=grp)
axes[1].legend(markerscale=5); axes[1].set_title("Spatial footprint by observer group")
plt.tight_layout()

### 2.8 Licensing

In [ ]:
inat["license_code"].value_counts(dropna=False).to_frame("n")

## Summary — citizen-science-specific facts

- **Presence-only**: no absence, abundance or cover information.
- Temporal distribution very uneven — 2021 alone is the largest year (43%
  of records); little data before 2019.
- A **single observer** produced the large majority of records (83.2%),
  and almost all `casual`-grade records — quality grade and observer
  identity are confounded here. This single-region,
  single-observer-dominated pilot cannot yet support a genuine RQ3
  geographic comparison (needs a second, distinct region — see
  `data/README.md`, Not Yet Obtained).
- Many records share a 10 m pixel (37.2% share one with at least one other
  record), and a small number of 1 km cells hold a large share of all
  observations — strong spatial clustering, relevant to the RQ2
  distribution-improving filter (see `docs/research_questions.md`, RQ2).
- `positional_accuracy` is missing for 79.8% of records, so coordinate
  error relative to a 10 m pixel can't be assessed for most of the data.
- Only 11.7% of observations fall inside the downloaded TESSERA tiles
  (see `../04_data_integration/`), because the two pilots use different
  boundaries.

Open questions and missing data are tracked in `docs/` and
`data/README.md`, not duplicated here.